In [ ]:
!pip install databricks-sdk -U
dbutils.library.restartPython()

In [ ]:
import json
import re
import requests
import os
from databricks.sdk import WorkspaceClient
from typing import Optional, Tuple

# ========================================
# Widget Parameters (for Job/CI-CD integration)
# ========================================
dbutils.widgets.text("space_id", "", "Space ID (empty = create new)")
dbutils.widgets.text("input_file", "./genie_definition/genie_space.json", "Input JSON File (Dev)")
dbutils.widgets.text("output_file", "", "Output JSON File (Prod) - optional")
dbutils.widgets.text("warehouse_id", "", "Warehouse ID (required)")
dbutils.widgets.text("title", "", "Space Title")
dbutils.widgets.text("source_catalog", "", "Source Catalog")
dbutils.widgets.text("target_catalog", "", "Target Catalog")
dbutils.widgets.text("source_schema", "", "Source Schema")
dbutils.widgets.text("target_schema", "", "Target Schema")

# ========================================
# Get parameter values
# ========================================
SPACE_ID = dbutils.widgets.get("space_id") or None
INPUT_FILE = dbutils.widgets.get("input_file")
OUTPUT_FILE = dbutils.widgets.get("output_file") or None
WAREHOUSE_ID = dbutils.widgets.get("warehouse_id") or None
TITLE = dbutils.widgets.get("title") or None
SOURCE_CATALOG = dbutils.widgets.get("source_catalog") or None
TARGET_CATALOG = dbutils.widgets.get("target_catalog") or None
SOURCE_SCHEMA = dbutils.widgets.get("source_schema") or None
TARGET_SCHEMA = dbutils.widgets.get("target_schema") or None

# Generate default output file name if not provided
if not OUTPUT_FILE and SOURCE_CATALOG and TARGET_CATALOG:
    base, ext = os.path.splitext(INPUT_FILE)
    OUTPUT_FILE = f"{base}_prod{ext}"

# Determine operation mode
OPERATION = "UPDATE" if SPACE_ID else "CREATE"

# Validate required parameters
if OPERATION == "CREATE":
    if not WAREHOUSE_ID:
        raise ValueError("warehouse_id is required when creating a new Genie space.")
    if not TITLE:
        raise ValueError("title is required when creating a new Genie space.")

print(f"🚀 Operation Mode: {OPERATION}")
if SPACE_ID:
    print(f"   Space ID: {SPACE_ID}")

# ========================================
# Catalog/Schema Replacement Functions
# ========================================
def replace_catalog_schema(text: str, 
                           source_catalog: str, 
                           target_catalog: str,
                           source_schema: Optional[str] = None,
                           target_schema: Optional[str] = None) -> str:
    """
    Replace catalog and schema names in a text string.
    Handles both formats:
    - Without backticks: catalog.schema.table
    - With backticks: `catalog`.`schema`.`table`
    """
    result = text
    
    if source_schema and target_schema:
        # Replace backtick-quoted format: `catalog`.`schema`.
        pattern_backticks = rf'`{re.escape(source_catalog)}`\.`{re.escape(source_schema)}`\.'
        replacement_backticks = f'`{target_catalog}`.`{target_schema}`.'
        result = re.sub(pattern_backticks, replacement_backticks, result)
        
        # Replace non-quoted format: catalog.schema.
        pattern_plain = rf'\b{re.escape(source_catalog)}\.{re.escape(source_schema)}\.'
        replacement_plain = f'{target_catalog}.{target_schema}.'
        result = re.sub(pattern_plain, replacement_plain, result)
    else:
        # Replace backtick-quoted format: `catalog`.
        pattern_backticks = rf'`{re.escape(source_catalog)}`\.'
        replacement_backticks = f'`{target_catalog}`.'
        result = re.sub(pattern_backticks, replacement_backticks, result)
        
        # Replace non-quoted format: catalog.
        pattern_plain = rf'\b{re.escape(source_catalog)}\.'
        replacement_plain = f'{target_catalog}.'
        result = re.sub(pattern_plain, replacement_plain, result)
    
    return result


def update_genie_space_catalog(data: dict,
                               source_catalog: str,
                               target_catalog: str,
                               source_schema: Optional[str] = None,
                               target_schema: Optional[str] = None) -> Tuple[dict, int]:
    """Update all catalog/schema references in a Genie space JSON structure."""
    updated_data = json.loads(json.dumps(data))  # Deep copy
    updates_count = 0
    
    # Update data_sources.tables[].identifier
    if 'data_sources' in updated_data and 'tables' in updated_data['data_sources']:
        for table in updated_data['data_sources']['tables']:
            if 'identifier' in table:
                old_value = table['identifier']
                new_value = replace_catalog_schema(
                    old_value, source_catalog, target_catalog, source_schema, target_schema
                )
                if old_value != new_value:
                    table['identifier'] = new_value
                    updates_count += 1
    
    # Update data_sources.metric_views[].identifier
    if 'data_sources' in updated_data and 'metric_views' in updated_data['data_sources']:
        for metric_view in updated_data['data_sources']['metric_views']:
            if 'identifier' in metric_view:
                old_value = metric_view['identifier']
                new_value = replace_catalog_schema(
                    old_value, source_catalog, target_catalog, source_schema, target_schema
                )
                if old_value != new_value:
                    metric_view['identifier'] = new_value
                    updates_count += 1
    
    # Update instructions.example_question_sqls[].sql[]
    if 'instructions' in updated_data and 'example_question_sqls' in updated_data['instructions']:
        for example in updated_data['instructions']['example_question_sqls']:
            if 'sql' in example:
                for j, sql_line in enumerate(example['sql']):
                    old_value = sql_line
                    new_value = replace_catalog_schema(
                        old_value, source_catalog, target_catalog, source_schema, target_schema
                    )
                    if old_value != new_value:
                        example['sql'][j] = new_value
                        updates_count += 1
    
    # Update benchmarks.questions[].answer[].content[]
    if 'benchmarks' in updated_data and 'questions' in updated_data['benchmarks']:
        for question in updated_data['benchmarks']['questions']:
            if 'answer' in question:
                for answer in question['answer']:
                    if 'content' in answer:
                        for k, content_line in enumerate(answer['content']):
                            old_value = content_line
                            new_value = replace_catalog_schema(
                                old_value, source_catalog, target_catalog, source_schema, target_schema
                            )
                            if old_value != new_value:
                                answer['content'][k] = new_value
                                updates_count += 1
    
    return updated_data, updates_count


# ========================================
# Initialize client and get auth headers
# ========================================
w = WorkspaceClient()
workspace_url = w.config.host

# Use SDK's built-in auth (works on serverless, clusters, and all auth types)
headers = w.config.authenticate()
headers["Content-Type"] = "application/json"

# Debug: print auth type being used
print(f"🔐 Auth type: {w.config.auth_type}")
print(f"🌐 Workspace URL: {workspace_url}")

# ========================================
# Load the original space definition (Dev)
# ========================================
with open(INPUT_FILE, 'r') as f:
    genie_space_data = json.load(f)

print(f"\n📄 Loaded space definition from: {INPUT_FILE}")

# ========================================
# Apply catalog/schema replacement if configured
# ========================================
if SOURCE_CATALOG and TARGET_CATALOG:
    source_display = SOURCE_CATALOG + (f".{SOURCE_SCHEMA}" if SOURCE_SCHEMA else "")
    target_display = TARGET_CATALOG + (f".{TARGET_SCHEMA}" if TARGET_SCHEMA else "")
    
    print(f"\n🔄 Replacing catalog/schema names:")
    print(f"   Source: {source_display}")
    print(f"   Target: {target_display}")
    
    genie_space_data, updates_count = update_genie_space_catalog(
        genie_space_data,
        source_catalog=SOURCE_CATALOG,
        target_catalog=TARGET_CATALOG,
        source_schema=SOURCE_SCHEMA,
        target_schema=TARGET_SCHEMA
    )
    print(f"   ✓ Made {updates_count} replacements")
    
    # Save the modified JSON to the output file (Prod version)
    if OUTPUT_FILE:
        with open(OUTPUT_FILE, 'w') as f:
            json.dump(genie_space_data, f, indent=2)
        print(f"\n💾 Saved Prod version to: {OUTPUT_FILE}")
        print(f"   (Original Dev version preserved at: {INPUT_FILE})")

# Convert to JSON string for API
genie_space_json_str = json.dumps(genie_space_data)

# ========================================
# Deploy: Create or Update
# ========================================
if OPERATION == "CREATE":
    # CREATE new Genie space
    payload = {
        "serialized_space": genie_space_json_str,
        "warehouse_id": WAREHOUSE_ID,
        "title": TITLE
    }
    
    print(f"\n📤 Creating new Genie space...")
    response = requests.post(
        f"{workspace_url}/api/2.0/genie/spaces",
        headers=headers,
        data=json.dumps(payload)
    )
    
    # Print response details for debugging
    print(f"   Response status: {response.status_code}")
    if response.status_code != 200:
        print(f"   Response body: {response.text}")
    
    response.raise_for_status()
    space_info = response.json()
    
    print(f"\n✅ Successfully CREATED Genie space:")
    print(f"   - Space ID: {space_info['space_id']}")
    print(f"   - Title: {space_info['title']}")
    
    # Important: Output the new space_id for future runs
    print(f"\n⚠️  IMPORTANT: Save this Space ID for future updates:")
    print(f"   space_id = \"{space_info['space_id']}\"")
    
    # Set the space_id as notebook output for downstream tasks
    dbutils.notebook.exit(json.dumps({
        "status": "created",
        "space_id": space_info['space_id'],
        "title": space_info['title']
    }))

else:
    # UPDATE existing Genie space
    payload = {
        "serialized_space": genie_space_json_str
    }
    
    # Add optional overrides
    if TITLE:
        payload["title"] = TITLE
    if WAREHOUSE_ID:
        payload["warehouse_id"] = WAREHOUSE_ID
    
    print(f"\n📤 Updating existing Genie space...")
    response = requests.patch(
        f"{workspace_url}/api/2.0/genie/spaces/{SPACE_ID}",
        headers=headers,
        data=json.dumps(payload)
    )
    
    # Print response details for debugging
    print(f"   Response status: {response.status_code}")
    if response.status_code != 200:
        print(f"   Response body: {response.text}")
    
    response.raise_for_status()
    space_info = response.json()
    
    print(f"\n✅ Successfully UPDATED Genie space:")
    print(f"   - Space ID: {space_info.get('space_id', SPACE_ID)}")
    print(f"   - Title: {space_info.get('title', 'N/A')}")
    
    # Set output for downstream tasks
    dbutils.notebook.exit(json.dumps({
        "status": "updated",
        "space_id": space_info.get('space_id', SPACE_ID),
        "title": space_info.get('title', 'N/A')
    }))

# ========================================
# Summary
# ========================================
print("\n🎉 Deployment complete!")

if SOURCE_CATALOG and TARGET_CATALOG and OUTPUT_FILE:
    print("\n" + "="*50)
    print("📁 Files:")
    print(f"   Dev version:  {INPUT_FILE}")
    print(f"   Prod version: {OUTPUT_FILE}")
    print("="*50)